# Binding - OLMo 2 Model Family

### Olmo 2 - 1B
model_name = OLMo-2-0425-1B  
revision = stage1-step1907359-tokens4001B

### Olmo 2 - 7B
model_name = OLMo-2-1124-7B  
revision = stage1-step928646-tokens3896B

### Olmo 2 - 13B
model_name = OLMo-2-1124-13B  
revision = stage1-step596057-tokens5001B

## Setup

In [ ]:
# Cell 0: Environment Detection
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

In [ ]:
# Cell 1: Colab Only — Install pinned dependencies
# ⚠️ Restart runtime after running this cell, then skip to Cell 2
if IN_COLAB:
    %pip install -q transformer_lens==2.18.0
    %pip install -q numpy==1.26.4
    %pip install -q transformers==4.57.6

In [ ]:
# Cell 1a: Confirm Transformer Lens version
from importlib.metadata import version
print("TransformerLens version:", version("transformer-lens"))

In [ ]:
# Cell 1b: Environment check after session restart
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

In [ ]:
# Cell 2: Project Root & Path Setup
if IN_COLAB:
    from google.colab import userdata
    token = userdata.get("GH_TMLR")

    repo_owner = "trishasalas"
    repo_name = "tmlr"
    repo_url = f"https://{token}@github.com/{repo_owner}/{repo_name}.git"

    PROJECT_ROOT = Path("/content") / repo_name

    if not PROJECT_ROOT.exists():
        !git clone {repo_url} {PROJECT_ROOT}
else:
    # Local: notebook lives in notebooks/, project root is one level up
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Cell 3: Imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Added {PROJECT_ROOT} to sys.path")

import torch
import src
from transformer_lens import HookedTransformer
import transformer_lens.utils as utils

# Device selection
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

In [ ]:
import src
from src.olmo_config import OLMO_REVISIONS
from src.tl217_olmo2_adapter import load_olmo2_tl217

In [ ]:
# Cell 5 - Model name variable
model_name = "OLMo-2-0425-1B"
revision = OLMO_REVISIONS[model_name]

In [ ]:
# Cell 6: Load Model
model = load_olmo2_tl217(f"allenai/{model_name}", device=device, revision=revision)

print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

In [ ]:
# Binding Battery — Loads compound pairs from YAML, measures attention from the
# second constituent back to the first across every layer and head, and saves
# per-domain CSVs plus a run manifest.
from pathlib import Path
import torch
import yaml
import pandas as pd
from src.manifest import write_binding_manifest


def find_token_index(tokens, target):
    """
    Find the index of a target word in a list of tokens.
    Handles leading-space tokenization (e.g., ' screen' for 'screen')
    and subword splits (e.g., 'keyboard' -> ['Key', 'board']).

    For multi-token matches, returns the LAST subtoken index — that's
    where the composed representation lives after the model processes
    the subword sequence.

    Returns the index or None if not found.
    """
    target_lower = target.lower()

    # Pass 1: exact single-token match (stripped of whitespace)
    for i, tok in enumerate(tokens):
        if tok.strip().lower() == target_lower:
            return i

    # Pass 2: multi-token match — concatenate adjacent tokens
    for start in range(len(tokens)):
        concat = ""
        for end in range(start, len(tokens)):
            concat += tokens[end].strip().lower()
            if concat == target_lower:
                # Return the LAST token in the span
                return end
            if len(concat) > len(target_lower):
                break

    return None


prompt_files = [
    'control.yaml',
    'accessibility.yaml',
    'medical.yaml',
    'legal.yaml',
    'finance.yaml',
]

all_results = []
domain_counts = {}
unresolved = []

output_dir = PROJECT_ROOT / 'results' / 'binding' / 'gpt2' / model_name
output_dir.mkdir(parents=True, exist_ok=True)

for prompts_file in prompt_files:
    domain = Path(prompts_file).stem
    prompts_path = PROJECT_ROOT / 'data' / 'binding' / prompts_file
    with open(prompts_path, 'r') as f:
        templates = yaml.safe_load(f)
    compounds = templates['compounds']
    print(f"\n--- Running {domain}: {len(compounds)} compounds ---")

    results = []
    for i, case in enumerate(compounds):
        print(f"\r  {i+1}/{len(compounds)}", end="")
        name = case['name']
        word1, word2, prompt = case['word1'], case['word2'], case['prompt']

        tokens = model.to_str_tokens(prompt)
        idx1 = find_token_index(tokens, word1)
        idx2 = find_token_index(tokens, word2)

        if idx1 is None or idx2 is None:
            unresolved.append((domain, name, word1, word2))
            print(f"\n  WARNING: could not locate '{word1}' / '{word2}' in {name}")
            continue

        # word2 attends back to word1 (e.g. "reader" -> "screen")
        target_idx = max(idx1, idx2)   # later token
        source_idx = min(idx1, idx2)   # earlier token

        # Only attention patterns are read — caching every activation wastes
        # a lot of memory on the larger models.
        with torch.no_grad():
            _, cache = model.run_with_cache(
                prompt, names_filter=lambda n: n.endswith("pattern")
            )

        for layer in range(model.cfg.n_layers):
            attention = cache["pattern", layer]   # [batch, heads, seq, seq]
            for head in range(model.cfg.n_heads):
                score = attention[0, head, target_idx, source_idx].item()
                results.append({
                    'compound': name,
                    'layer': layer,
                    'head': head,
                    'binding_score': round(score, 4),
                    'word1': word1,
                    'word2': word2,
                    'prompt': prompt,
                    'tokens': str(tokens),
                    'word1_idx': source_idx,
                    'word2_idx': target_idx,
                    'domain': domain,
                    'model': model_name,
                })

        del cache
    print()

    domain_df = pd.DataFrame(results)
    filename = f'{model_name}-{domain}.csv'
    domain_df.to_csv(output_dir / filename, index=False)
    all_results.append(domain_df)

    domain_counts[domain] = {
        'expected': len(compounds) * model.cfg.n_layers * model.cfg.n_heads,
        'written': len(domain_df),
        'file': filename,
    }
    print(f"  → {filename}  ({len(domain_df)} rows)")

results_df = pd.concat(all_results, ignore_index=True)

write_binding_manifest(
    PROJECT_ROOT, output_dir, model_name, model,
    domain_counts, results_df, prompt_files,
)

if unresolved:
    print(f"\n⚠️  {len(unresolved)} compounds unresolved (skipped):")
    for d, n, w1, w2 in unresolved:
        print(f"    {d}/{n}: '{w1}' / '{w2}'")

print(f"\nSaved {len(results_df)} rows across {len(all_results)} domains → {output_dir}")


### Delete Model & Clear Cache

In [ ]:
# Cell 7: Free memory for next model
import gc
del model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    print(f"Memory cleared — GPU: {torch.cuda.memory_allocated()/1e9:.1f}GB allocated")
elif device == "mps":
    torch.mps.empty_cache()
    print("Memory cleared")